In [1]:
import pandas as pd
from math import ceil
import os
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.multioutput import MultiOutputClassifier
import random
from sklearn.utils import resample

SEED =42 


random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
data = pd.read_csv("raw will used/genuine test/3.csv")
data.info()

In [ ]:
data["16"].value_counts()

In [ ]:
data[data["16"].isin([33286, 33285])]

In [ ]:
marker_values = [33285.0, 33286.0]
marker_idx = data[data["16"].isin(marker_values)].index

# Markerlar arası fark
diffs = marker_idx.to_series().diff()
diffs.describe()

In [ ]:


target = 33285
non_target = 33286 

my_target = 1
my_nontarget = 0
window = 0.8 # 0.8 seconds
sample = 512 # 512 hertz
sample_count = ceil(sample*window)

markers = data[data["16"].isin([non_target, target])][["Unnamed: 0", "16"]]

refined = []




In [ ]:
new_data=None
for marker_id , label in markers[["Unnamed: 0","16"]].itertuples(index=False):
    intlabel = int(label)
    new_data = data.iloc[marker_id:marker_id+sample_count, 1:17].to_numpy()
    new_data = new_data.T
    new_data = pd.DataFrame(new_data)
    print(type(new_data))
    break

print(new_data)



In [ ]:

save_folder = "data_files/"

target = 33285
non_target = 33286 
my_target = 1
my_nontarget = 0
window = 0.8 # 0.8 seconds
sample = 512 # 512 hertz
sample_count = ceil(sample*window)

#markers = data[data["16"].isin([non_target, target])][["Unnamed: 0", "16"]]

path_list =["raw will used/genuine train" , "raw will used/imposter train"]
imposter = 0
genuine =1
counter =1
person = 1
for path in path_list:
    files = os.listdir(path)
    for file in files :
        tdata = pd.read_csv(path+"/"+file)
        markers = tdata[tdata["16"].isin([non_target, target])][["Unnamed: 0", "16"]]
        for marker_id , label in markers[["Unnamed: 0","16"]].itertuples(index=False):
            intlabel = int(label)
            target_label  = 1 if intlabel==target else 0
            new_data = tdata.iloc[marker_id:marker_id+sample_count, 1:17].to_numpy()
            new_data = new_data.T
            new_data = pd.DataFrame(new_data)
            file_name = save_folder + str(counter) + "_" + str(person) + "_"  + str(target_label) + ".csv"
            new_data.to_csv(file_name,index=False)
            counter += 1        
    person = 0

In [ ]:
# Kaydedilecek klasör
save_folder = "data_files/"
os.makedirs(save_folder, exist_ok=True)  # klasör yoksa oluştur

# Parametreler
target = 33285
non_target = 33286
window = 0.8  # saniye
sample_rate = 512  # Hz
sample_count = ceil(sample_rate * window)

# Dosya yolları
path_list = ["raw will used/genuine train", "raw will used/imposter train"]

counter = 1
person = 1  # genuine=1, imposter=0

for path in path_list:
    files = os.listdir(path)
    
    for file in files:
        tdata = pd.read_csv(os.path.join(path, file))
        
        # Sadece target ve non-target markerlar
        markers = tdata[tdata["16"].isin([non_target, target])][["Unnamed: 0", "16"]]
        
        for marker_id, label in markers.itertuples(index=False):
            marker_id = int(marker_id)  # güvenlik için integer
            
            # Eğer window veri dışında kalıyorsa atla
            if marker_id + sample_count > len(tdata):
                continue
            
            # Target label 1=target, 0=non-target
            target_label = 1 if int(label) == target else 0
            # Segmenti al ve transpose et (CNN input için channel-first)
            new_data = tdata.iloc[marker_id:marker_id+sample_count, 1:17].to_numpy()
            new_data = new_data.T
            new_data_df = pd.DataFrame(new_data)
            
            # Dosya adı: counter_person_target.csv
            file_name = os.path.join(save_folder, f"{counter}_{person}_{target_label}.csv")
            new_data_df.to_csv(file_name, index=False)
            
            counter += 1
    
    # Sıradaki klasör için person=0 (imposter)
    person = 0

In [ ]:
trainfiles = os.listdir("data_files")
X_list = []
Y_list_person = []
Y_list_target = []
for mmfile in trainfiles:
    base_name = mmfile.replace(".csv", "")
    parts = base_name.split("_")
    person  = int(parts[1])     # 0 → imposter
    target_label =0 if person ==0 else int(parts[2])
    df = pd.read_csv(os.path.join("data_files", mmfile))
    X_list.append(df.to_numpy().flatten())
    Y_list_person.append(person) 
    Y_list_target.append(target_label)  

X = np.array(X_list)
Y_person = np.array(Y_list_person) 
Y_target = np.array(Y_list_target) 
scaler = StandardScaler()
scaler2 = StandardScaler()

X_train_person, X_test_person, Y_train_person, Y_test_person = train_test_split(
    X, Y_person, 
    test_size=0.2, 
    random_state=SEED,
    stratify=Y_person
)
X_train_person = scaler.fit_transform(X_train_person)
X_test_person = scaler.transform(X_test_person)
X_train_target, X_test_target, Y_train_target, Y_test_target = train_test_split(
    X, Y_target, 
    test_size=0.2, 
    random_state=SEED,
    stratify=Y_target 
)

X_train_target = scaler2.fit_transform(X_train_target)
X_test_target = scaler2.transform(X_test_target)


idx_0 = np.where(Y_train_target == 0)[0]
idx_1 = np.where(Y_train_target == 1)[0]

n_min = len(idx_1)

idx_0_down = resample(
    idx_0,
    replace=False,
    n_samples=n_min,
    random_state=SEED
)

idx_balanced = np.concatenate([idx_0_down, idx_1])

X_train_target = X_train_target[idx_balanced]
Y_train_target = Y_train_target[idx_balanced]


### Plasticity ve connectome degisimleri olmadan classification yaptigi sonuclar

In [ ]:
person_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=4,          # 6'dan 4'e
    l2_leaf_reg=10,   # regularization ekle
    verbose=100,
    random_seed=SEED
)

# Target için class_weights ile azınlık sınıfına ağırlık
# Örnek: non-target=1, target=5
target_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=4,          # 6'dan 4'e
    l2_leaf_reg=10,   # regularization ekle
    verbose=100,
    random_seed=SEED
)


person_model.fit(X_train_person, Y_train_person)
target_model.fit(X_train_target, Y_train_target)
person_pred = person_model.predict(X_test_person)
target_pred = target_model.predict(X_test_target)

# Accuracy ayrı ayrı hesapla

print("Person accuracy:", accuracy_score(Y_test_person, person_pred))
print("Target accuracy:", accuracy_score(Y_test_target, target_pred))
print(classification_report(Y_test_target, target_pred))


In [ ]:
#print(classification_report(Y_test[:,0], Y_pred[:,0]))
print(classification_report(Y_test_person, person_pred))

### Plasticity ve connectome degisimi ile sonuclar

In [ ]:

# Kaydedilecek klasör
save_folder = "data_files2/"
os.makedirs(save_folder, exist_ok=True)  # klasör yoksa oluştur

# Parametreler
target = 33285
non_target = 33286
window = 0.8  # saniye
sample_rate = 512  # Hz
sample_count = ceil(sample_rate * window)

# Dosya yolları
path_list = ["raw will used/genuine test", "raw will used/imposter test"]

counter = 1
person = 1  # genuine=1, imposter=0

for path in path_list:
    files = os.listdir(path)
    
    for file in files:
        tdata = pd.read_csv(os.path.join(path, file))
        
        # Sadece target ve non-target markerlar
        markers = tdata[tdata["16"].isin([non_target, target])][["Unnamed: 0", "16"]]
        
        for marker_id, label in markers.itertuples(index=False):
            marker_id = int(marker_id)  # güvenlik için integer
            
            # Eğer window veri dışında kalıyorsa atla
            if marker_id + sample_count > len(tdata):
                continue
            
            # Target label 1=target, 0=non-target
            target_label = 1 if int(label) == target else 0
            # Segmenti al ve transpose et (CNN input için channel-first)
            new_data = tdata.iloc[marker_id:marker_id+sample_count, 1:17].to_numpy()
            new_data = new_data.T
            new_data_df = pd.DataFrame(new_data)
            
            # Dosya adı: counter_person_target.csv
            file_name = os.path.join(save_folder, f"{counter}_{person}_{target_label}.csv")
            new_data_df.to_csv(file_name, index=False)
            
            counter += 1
    
    # Sıradaki klasör için person=0 (imposter)
    person = 0

In [ ]:



trainfiles = os.listdir("data_files2")
X_list2 = []
Y_list2_person = []
Y_list2_target = []
for mmfile in trainfiles:
    base_name = mmfile.replace(".csv", "")
    parts = base_name.split("_")
    person  = int(parts[1])     # 0 -> imposter
    target_label = 0 if person ==0 else int(parts[2])# 0 -> non-target
    df = pd.read_csv(os.path.join("data_files2", mmfile))
    X_list2.append(df.to_numpy().flatten())
    Y_list2_person.append(person) 
    Y_list2_target.append(target_label)  

X2 = np.array(X_list2)
Y2_person = np.array(Y_list2_person) 
Y2_target = np.array(Y_list2_target) 
X_scaled2_person = scaler.transform(X2)
X_scaled2_target = scaler2.transform(X2)


person_pred2 = person_model.predict(X_scaled2_person)
target_pred2 = target_model.predict(X_scaled2_target)

# Accuracy ayrı ayrı hesapla

print("Person accuracy:", accuracy_score(Y2_person, person_pred2))
print("Target accuracy:", accuracy_score(Y2_target, target_pred2))
print(classification_report(Y2_target, target_pred2))


In [ ]:
#print(classification_report(Y2[:,0], Y_pred2[:,0]))
print(classification_report(Y2_person , person_pred2))

In [ ]:
np.unique(person_pred2)

In [ ]:
np.unique(target_pred2)

In [ ]:
print("Train target dağılımı:", np.bincount(Y_target))
print("Test2 target dağılımı:", np.bincount(Y2_target))
print("Train person dağılımı:", np.bincount(Y_person))
print("Test2 person dağılımı:", np.bincount(Y2_person))